# Figure 1 — Combined: Study Area Map + Discharge Distribution
**Target journal:** Computers and Geosciences  
**Layout:** (a) Ebro River Basin study area map  |  (b) KDE + histogram of log₁₀(Qmean)  
**Output:** `output/figures/Figure1_Combined.png` at 300 DPI (full-page width, ~254 mm × 140 mm)

In [ ]:
# =============================================================================
# CELL 1 — Imports & global rcParams
# =============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LightSource, Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry import mapping

# ── Global style — identical across both panels ───────────────────────────────
mpl.rcParams.update({
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size'          : 8,
    'axes.linewidth'     : 0.7,
    'xtick.major.width'  : 0.7,
    'ytick.major.width'  : 0.7,
    'xtick.minor.width'  : 0.4,
    'ytick.minor.width'  : 0.4,
    'xtick.major.size'   : 4,
    'ytick.major.size'   : 4,
    'xtick.minor.size'   : 2,
    'ytick.minor.size'   : 2,
    'xtick.direction'    : 'out',
    'ytick.direction'    : 'out',
    'figure.dpi'         : 150,    # screen preview
    'savefig.dpi'        : 300,    # publication output
})

print('All libraries loaded.')

In [ ]:
# =============================================================================
# CELL 2 — File paths
# =============================================================================
SPATIAL = os.path.join('data', 'spatial')

dem_path      = os.path.join(SPATIAL, 'Dem_bueno_fill.tif')
basin_path    = os.path.join(SPATIAL, 'Limite_Cuenca_Ebro.shp')
streams_path  = os.path.join(SPATIAL, 'Stream.shp')
stations_path = os.path.join(SPATIAL, 'Estaciones_finalfinal.shp')
csv_path      = os.path.join('data', 'SSI_daily.csv')
output_path   = os.path.join('output', 'figures', 'Figure1_Combined.png')

In [ ]:
# =============================================================================
# CELL 3 — Load & reproject vector data
# =============================================================================
basin    = gpd.read_file(basin_path)
streams  = gpd.read_file(streams_path)
stations = gpd.read_file(stations_path)

TARGET_EPSG  = 25830
basin_utm    = basin.to_crs(epsg=TARGET_EPSG)
streams_utm  = streams.to_crs(epsg=TARGET_EPSG)
stations_utm = stations.to_crs(epsg=TARGET_EPSG)
basin_ll     = basin.to_crs(epsg=4326)   # geographic, for inset map

xmin, ymin, xmax, ymax = basin_utm.total_bounds

# Station colour (single class — no regime column in this dataset)
COLOR_DEFAULT = '#d73027'
stations_utm  = stations_utm.copy()
stations_utm['_color'] = COLOR_DEFAULT

print(f'Basin extent  : X [{xmin/1e3:.0f}–{xmax/1e3:.0f}] km,  Y [{ymin/1e3:.0f}–{ymax/1e3:.0f}] km')
print(f'Stations      : {len(stations_utm)}')
print(f'Stream segs   : {len(streams_utm)}')

In [ ]:
# =============================================================================
# CELL 4 — Load DEM, mask to basin, compute hillshade & shaded relief
# =============================================================================
with rasterio.open(dem_path) as src:
    dem_crs    = src.crs
    dem_nodata = src.nodata
    dem_res_x  = abs(src.transform.a)
    dem_res_y  = abs(src.transform.e)
    basin_dem  = basin.to_crs(dem_crs)
    geoms      = [mapping(g) for g in basin_dem.geometry]
    arr, masked_transform = rio_mask(src, geoms, crop=True, nodata=-9999, filled=True)

dem_raw = arr[0].astype(np.float32)
dem_raw[np.isclose(dem_raw, -9999, atol=1)] = np.nan
dem_raw[dem_raw < -500] = np.nan

rows, cols = dem_raw.shape
west  = masked_transform.c
north = masked_transform.f
east  = west  + cols * masked_transform.a
south = north + rows * masked_transform.e

# Hillshade + shaded relief blend
ls         = LightSource(azdeg=315, altdeg=40)
dem_filled = np.where(np.isnan(dem_raw), 0.0, dem_raw)
vmin_dem   = np.nanmin(dem_raw)
vmax_dem   = np.nanmax(dem_raw)

# Custom terrain cmap (no ocean blue)
_tc      = plt.cm.terrain(np.linspace(0.22, 1.0, 256))
TERRAIN  = LinearSegmentedColormap.from_list('terrain_land', _tc, N=256)

rgb_shaded = ls.shade(dem_filled, cmap=TERRAIN, blend_mode='overlay',
                      vert_exag=1.8, dx=dem_res_x, dy=dem_res_y,
                      vmin=vmin_dem, vmax=vmax_dem)
rgb_shaded[np.isnan(dem_raw), 3] = 0.0   # transparent outside basin

print(f'DEM      : {rows}×{cols} px,  {vmin_dem:.0f}–{vmax_dem:.0f} m a.s.l.')
print(f'RGBA img : {rgb_shaded.shape}')

In [ ]:
# =============================================================================
# CELL 5 — Load discharge data & compute KDE / statistics
# =============================================================================
df       = pd.read_csv(csv_path, parse_dates=['date'])
df_valid = df[df['Q'] > 0].copy()   # discard zeros before aggregating

qmean = (df_valid
         .groupby('station_id')['Q']
         .mean()
         .rename('Qmean')
         .reset_index())
qmean['log10_Qmean'] = np.log10(qmean['Qmean'])

x_vals     = qmean['log10_Qmean'].values
n_stations = len(x_vals)

# KDE
kde    = gaussian_kde(x_vals, bw_method='scott')
x_pad  = 0.4
x_grid = np.linspace(x_vals.min() - x_pad, x_vals.max() + x_pad, 500)
y_grid = kde(x_grid)

# Summary statistics
x_median = np.median(x_vals)
x_min    = x_vals.min()
x_max    = x_vals.max()
x_p25    = np.percentile(x_vals, 25)
x_p75    = np.percentile(x_vals, 75)

idx_min   = qmean['log10_Qmean'].idxmin()
idx_max   = qmean['log10_Qmean'].idxmax()
qmean_min = qmean.loc[idx_min, 'Qmean']
qmean_max = qmean.loc[idx_max, 'Qmean']

# Histogram bin count (Freedman-Diaconis, clamped to 5–8 for n=33)
iqr_val      = x_p75 - x_p25
bin_width_fd = 2 * iqr_val / (n_stations ** (1/3))
n_bins       = max(5, min(8, int(np.ceil((x_max - x_min) / bin_width_fd))))

print(f'Stations  : {n_stations}')
print(f'Qmean range : {qmean_min:.2f} – {qmean_max:.1f} m³ s⁻¹')
print(f'Median (log): {x_median:.3f}  →  {10**x_median:.1f} m³ s⁻¹')
print(f'Histogram bins: {n_bins}')

In [ ]:
# =============================================================================
# CELL 6 — Build combined Figure 1
# =============================================================================
#
#  Layout (figure-fraction coordinates [left, bottom, width, height]):
#
#  ┌────────────────────────────────────────────────────────────┬──┬──────────────────────┐
#  │                                                            │  │   (b) DISTRIBUTION   │
#  │                    (a) MAP AXES                            │CB│                      │
#  │              [0.040, 0.065, 0.500, 0.880]                  │  │  [0.675, 0.090,       │
#  │                                                            │  │   0.310, 0.760]      │
#  └────────────────────────────────────────────────────────────┴──┴──────────────────────┘
#
#  Total: 10.0 × 5.5 inches (254 × 140 mm)
#  NOTE: tight_layout() is NOT called — all axes are manually positioned to
#        avoid conflicts between cartopy GeoAxes and regular matplotlib axes.

FIG_W = 10.0   # inches  (254 mm)
FIG_H = 5.50   # inches  (140 mm)

utm30_proj = ccrs.UTM(zone=30)
geo_proj   = ccrs.PlateCarree()

fig = plt.figure(figsize=(FIG_W, FIG_H))

# ═════════════════════════════════════════════════════════════════════════════
# PANEL (a) — Study area map
# ═════════════════════════════════════════════════════════════════════════════
PAD    = 15_000   # 15 km padding around basin
ax_map = fig.add_axes([0.040, 0.065, 0.500, 0.880], projection=utm30_proj)
ax_map.set_extent([xmin - PAD, xmax + PAD, ymin - PAD, ymax + PAD], crs=utm30_proj)
ax_map.set_facecolor('#e8f4f8')

# ── DEM shaded relief ────────────────────────────────────────────────────────
dem_plot_crs = utm30_proj if dem_crs.to_epsg() == 25830 else geo_proj
ax_map.imshow(rgb_shaded, extent=[west, east, south, north],
              transform=dem_plot_crs, origin='upper',
              interpolation='bilinear', zorder=1)

# ── River network ─────────────────────────────────────────────────────────────
ax_map.add_geometries(streams_utm.geometry, crs=utm30_proj,
                      facecolor='none', edgecolor='#1a6eb5',
                      linewidth=0.35, zorder=3)

# ── Basin boundary ────────────────────────────────────────────────────────────
ax_map.add_geometries(basin_utm.geometry, crs=utm30_proj,
                      facecolor='none', edgecolor='black',
                      linewidth=1.4, zorder=4)

# ── Gauging stations ──────────────────────────────────────────────────────────
xs = [g.x for g in stations_utm.geometry]
ys = [g.y for g in stations_utm.geometry]
cs = list(stations_utm['_color'])
ax_map.scatter(xs, ys, c=cs, s=22, marker='^',
               edgecolors='black', linewidths=0.5,
               transform=utm30_proj, zorder=5)

# ── Gridlines (1° main + 0.5° sub) ───────────────────────────────────────────
gl = ax_map.gridlines(crs=geo_proj, draw_labels=True,
                      linewidth=0.55, color='#555555', alpha=0.7,
                      linestyle='--', zorder=6)
gl.top_labels = False; gl.right_labels = False
gl.xformatter = LONGITUDE_FORMATTER; gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 7.5, 'color': 'black', 'fontfamily': 'serif'}
gl.ylabel_style = {'size': 7.5, 'color': 'black', 'fontfamily': 'serif'}
gl.xlocator = mticker.FixedLocator(np.arange(-3, 5, 1))
gl.ylocator = mticker.FixedLocator(np.arange(40, 44, 1))

gl2 = ax_map.gridlines(crs=geo_proj, linewidth=0.25, color='#888888',
                        alpha=0.45, linestyle=':', zorder=6)
gl2.top_labels = gl2.right_labels = gl2.left_labels = gl2.bottom_labels = False
gl2.xlocator = mticker.FixedLocator(np.arange(-3.5, 5, 0.5))
gl2.ylocator = mticker.FixedLocator(np.arange(39.5, 44.5, 0.5))

# ── Scale bar (100 km, lower-right of map) ────────────────────────────────────
SB_LEN = 100_000
SB_X0  = xmax - SB_LEN - 25_000
SB_Y0  = ymin + 15_000
SB_H   = 6_000
SB_TY  = SB_Y0 - 12_000
for x_s, x_e, col in [(SB_X0, SB_X0 + SB_LEN // 2, 'black'),
                       (SB_X0 + SB_LEN // 2, SB_X0 + SB_LEN, 'white')]:
    ax_map.add_patch(mpatches.Rectangle(
        (x_s, SB_Y0 - SB_H // 2), x_e - x_s, SB_H,
        facecolor=col, edgecolor='black', linewidth=0.7,
        transform=utm30_proj, zorder=7))
for x_tick, lbl in [(SB_X0, '0'),
                    (SB_X0 + SB_LEN // 2, '50'),
                    (SB_X0 + SB_LEN, '100')]:
    ax_map.text(x_tick, SB_TY, lbl, ha='center', va='top',
                fontsize=6.5, fontfamily='serif', transform=utm30_proj, zorder=7)
ax_map.text(SB_X0 + SB_LEN * 0.5, SB_TY - 9_000, 'km',
            ha='center', va='top', fontsize=6.5, fontfamily='serif',
            transform=utm30_proj, zorder=7)

# ── North arrow ───────────────────────────────────────────────────────────────
ax_map.annotate('', xy=(0.955, 0.940), xytext=(0.955, 0.855),
                xycoords='axes fraction',
                arrowprops=dict(
                    arrowstyle='->, head_width=0.25, head_length=0.25',
                    fc='black', ec='black', linewidth=1.2), zorder=8)
ax_map.text(0.955, 0.955, 'N', transform=ax_map.transAxes,
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            fontfamily='serif', zorder=8)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], color='black', linewidth=1.4, label='Basin boundary'),
    Line2D([0], [0], color='#1a6eb5', linewidth=0.9, label='River network'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor=COLOR_DEFAULT,
           markeredgecolor='black', markeredgewidth=0.5,
           markersize=6, label='Gauging station'),
]
leg_map = ax_map.legend(
    handles=legend_elements, loc='center left',
    fontsize=7, framealpha=0.88, edgecolor='black',
    fancybox=False, borderpad=0.6, handlelength=1.8)
leg_map.get_title().set_fontfamily('serif')
leg_map.get_frame().set_linewidth(0.7)

# ── Inset location map — moved up (bottom: 0.230) ─────────────────────────────
ax_in = fig.add_axes([0.045, 0.200, 0.112, 0.210], projection=geo_proj)
ax_in.set_extent([-10.5, 4.5, 35.0, 44.5], crs=geo_proj)
ax_in.add_feature(cfeature.LAND.with_scale('50m'),
                  facecolor='#d9d9d9', edgecolor='none', zorder=1)
ax_in.add_feature(cfeature.OCEAN.with_scale('50m'),
                  facecolor='#c6e2f0', zorder=1)
ax_in.add_feature(cfeature.BORDERS.with_scale('50m'),
                  linewidth=0.5, edgecolor='#555555', zorder=2)
ax_in.add_feature(cfeature.COASTLINE.with_scale('50m'),
                  linewidth=0.5, edgecolor='#555555', zorder=2)
ax_in.add_geometries(basin_ll.geometry, crs=geo_proj,
                     facecolor=COLOR_DEFAULT, edgecolor='#7f0000',
                     linewidth=0.7, alpha=0.85, zorder=3)
for spine in ax_in.spines.values():
    spine.set_linewidth(0.7)
    spine.set_edgecolor('black')

# ── Elevation colorbar ────────────────────────────────────────────────────────
ax_cbar = fig.add_axes([0.553, 0.120, 0.015, 0.710])
norm = Normalize(vmin=vmin_dem, vmax=vmax_dem)
sm   = ScalarMappable(norm=norm, cmap=TERRAIN)
sm.set_array([])
cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical', extend='neither')
tick_step = 500 if (vmax_dem - vmin_dem) > 1500 else 250
tick_vals = np.arange(
    np.ceil(vmin_dem  / tick_step) * tick_step,
    np.floor(vmax_dem / tick_step) * tick_step + 1,
    tick_step)
cbar.set_ticks(tick_vals)
cbar.ax.tick_params(labelsize=7, length=3, width=0.6)
cbar.set_label('Elevation (m a.s.l.)', fontsize=8, labelpad=7, fontfamily='serif')

# ── Map frame ─────────────────────────────────────────────────────────────────
for spine in ax_map.spines.values():
    spine.set_linewidth(0.8)
    spine.set_edgecolor('black')

print('Panel (a) done.')

# ═════════════════════════════════════════════════════════════════════════════
# PANEL (b) — Discharge distribution  (shifted right: left=0.675)
# ═════════════════════════════════════════════════════════════════════════════
ax_dist = fig.add_axes([0.675, 0.090, 0.310, 0.760])

# ── Colour palette ────────────────────────────────────────────────────────────
C_HIST    = '#7ab5d4'
C_HIST_ED = '#3a85b0'
C_KDE     = '#1a4f72'
C_FILL    = '#c5dff3'
C_MEDIAN  = '#ca6f1e'
C_IQR     = '#aed6f1'
C_RUG     = '#2e86c1'
C_ANNOT   = '#1b2631'

# ── 1. Histogram ──────────────────────────────────────────────────────────────
ax_dist.hist(x_vals, bins=n_bins, density=True,
             color=C_HIST, edgecolor=C_HIST_ED, linewidth=0.6,
             alpha=0.70, zorder=2, label='Histogram')

# ── 2. IQR shaded band ────────────────────────────────────────────────────────
ax_dist.axvspan(x_p25, x_p75, color=C_IQR, alpha=0.35, zorder=1,
                label=f'IQR [{10**x_p25:.0f}–{10**x_p75:.0f} m³ s⁻¹]')

# ── 3. KDE fill + line ────────────────────────────────────────────────────────
ax_dist.fill_between(x_grid, y_grid, color=C_FILL, alpha=0.50, zorder=3)
ax_dist.plot(x_grid, y_grid, color=C_KDE, linewidth=1.5, zorder=4,
             label='KDE (Scott bandwidth)')

# ── 4. Median vertical line ───────────────────────────────────────────────────
ax_dist.axvline(x_median, color=C_MEDIAN, linewidth=1.4, linestyle='--', zorder=5,
                label=f'Median ({10**x_median:.1f} m³ s⁻¹)')

# ── 5. Rug plot ───────────────────────────────────────────────────────────────
for xv in x_vals:
    ax_dist.plot([xv, xv], [0, 0.030],
                 transform=ax_dist.get_xaxis_transform(),
                 color=C_RUG, linewidth=0.8, alpha=0.9, zorder=6, clip_on=True)

# ── 6. Min / Max annotations ──────────────────────────────────────────────────
y_text = 0.38
ax_dist.annotate(f'{qmean_min:.2f}\nm³ s⁻¹',
                 xy=(x_min, 0), xytext=(x_min + 0.05, y_text),
                 xycoords='data', textcoords='data',
                 fontsize=6.5, ha='left', va='top', color=C_ANNOT,
                 arrowprops=dict(arrowstyle='->', color=C_ANNOT, lw=0.7), zorder=7)
ax_dist.annotate(f'{qmean_max:.0f}\nm³ s⁻¹',
                 xy=(x_max, 0), xytext=(x_max - 0.05, y_text),
                 xycoords='data', textcoords='data',
                 fontsize=6.5, ha='right', va='top', color=C_ANNOT,
                 arrowprops=dict(arrowstyle='->', color=C_ANNOT, lw=0.7), zorder=7)

# ── 7. Axes labels & ticks ────────────────────────────────────────────────────
ax_dist.set_xlabel(r'Mean discharge  $\log_{10}$ (m³ s⁻¹)', fontsize=8)
ax_dist.set_ylabel('Density', fontsize=8)
ax_dist.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax_dist.tick_params(axis='both', which='major', labelsize=7)
ax_dist.tick_params(axis='x', which='minor', bottom=True)
ax_dist.xaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax_dist.set_xlim(x_vals.min() - x_pad * 0.8, x_vals.max() + x_pad * 0.8)

x_tick_log = np.arange(np.floor(x_min - 0.1), np.ceil(x_max + 0.2) + 0.5, 0.5)
ax_dist.set_xticks(x_tick_log)
ax_dist.set_xticklabels([f'{v:.1f}' for v in x_tick_log], fontsize=7)

ax_dist.spines['top'].set_visible(False)
ax_dist.spines['right'].set_visible(False)

# ── 8. Secondary x-axis — raw m³/s ────────────────────────────────────────────
ax2 = ax_dist.twiny()
ax2.set_xlim(ax_dist.get_xlim())
raw_ticks_m3s = [0.5, 1, 5, 10, 50, 100, 500, 1000]
log_ticks  = [np.log10(v) for v in raw_ticks_m3s
               if ax_dist.get_xlim()[0] <= np.log10(v) <= ax_dist.get_xlim()[1]]
raw_labels = [str(int(v)) if v >= 1 else str(v)
              for v in raw_ticks_m3s
              if ax_dist.get_xlim()[0] <= np.log10(v) <= ax_dist.get_xlim()[1]]
ax2.set_xticks(log_ticks)
ax2.set_xticklabels(raw_labels, fontsize=6.5)
ax2.set_xlabel(r'Mean discharge  (m³ s⁻¹)', fontsize=7, labelpad=4)
ax2.tick_params(axis='x', which='major', length=3, width=0.6, labelsize=6.5)
ax2.spines['right'].set_visible(False)

# ── 9. Legend ─────────────────────────────────────────────────────────────────
leg_dist = ax_dist.legend(
    loc='upper left', fontsize=6,
    framealpha=0.92, edgecolor='black',
    fancybox=False, borderpad=0.6, handlelength=1.4)
leg_dist.get_frame().set_linewidth(0.5)

# ── 10. Station count — moved to top right ────────────────────────────────────
ax_dist.text(0.97, 0.95, f'n = {n_stations} stations',
             transform=ax_dist.transAxes, ha='right', va='top',
             fontsize=6.5, color='black', style='italic')

print('Panel (b) done.')

# ─────────────────────────────────────────────────────────────────────────────
# PANEL LABELS (a) and (b)
# ─────────────────────────────────────────────────────────────────────────────
fig.text(0.010, 0.968, '(a)', fontsize=11, fontweight='bold',
         fontfamily='serif', va='top', ha='left')
fig.text(0.658, 0.968, '(b)', fontsize=11, fontweight='bold',
         fontfamily='serif', va='top', ha='left')

plt.show()
print('\nCombined figure rendered.')

In [ ]:
# =============================================================================
# CELL 7 — Export at 300 DPI
# =============================================================================
fig.savefig(
    output_path,
    dpi         = 300,
    bbox_inches = 'tight',
    pad_inches  = 0.05,
    facecolor   = 'white',
    transparent = False,
)
print(f'Saved → {output_path}')

from PIL import Image
img = Image.open(output_path)
w_mm = img.size[0] / 300 * 25.4
h_mm = img.size[1] / 300 * 25.4
print(f'Size  : {img.size[0]} × {img.size[1]} px')
print(f'       ({w_mm:.1f} × {h_mm:.1f} mm at 300 DPI)')
img.close()